In [11]:
import ollama
import json
import re
from pypdf import PdfReader

print("✅ Setup complete")

✅ Setup complete


In [12]:
reader = PdfReader(r"E:\PersonaForge-A-Multi-Agent-LLM-System-for-Intelligent-Interaction\me\Profile.pdf")

linkedin_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin_text += text

print("✅ LinkedIn data loaded")

✅ LinkedIn data loaded


In [13]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary_text = f.read()

print("✅ Summary loaded")

✅ Summary loaded


In [21]:
GENERATOR_MODEL = "llama3.2"
EVALUATOR_MODEL = "llama3.2"

NAME = "Nikith"  

SYSTEM_PROMPT = f"""
You are acting as {NAME}, an AI/Cloud Engineer.

You must answer like {NAME} based on real experience.

RULES:
- Use first person ("I", "my experience")
- Be professional and natural
- Keep answers concise
- If you don’t know, say so

--- SUMMARY ---
{summary_text}

--- LINKEDIN ---
{linkedin_text}
"""

In [22]:
EVALUATOR_PROMPT = """
You are a STRICT AI evaluator.

Evaluate the response based on:
- clarity
- correctness
- completeness
- professionalism
- instruction following
- human-like tone

IMPORTANT:
- If response is too long → simplify
- If not beginner-friendly → rewrite
- If robotic → make natural
- ALWAYS provide improved_response if possible

Return ONLY valid JSON.

Format:
{
  "score": 1-10,
  "verdict": "approve" or "improve",
  "feedback": "short feedback",
  "improved_response": "better response"
}
"""

In [23]:
def clean_response(text):
    parts = text.strip().split("\n\n")
    seen = []
    for p in parts:
        if p not in seen:
            seen.append(p)
    return "\n\n".join(seen)

In [24]:
def generator_agent(user_input, chat_history):
    user_input = user_input + "\n\nFollow the instruction strictly. Keep it short."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + chat_history + [
        {"role": "user", "content": user_input}
    ]

    response = ollama.chat(
        model=GENERATOR_MODEL,
        messages=messages
    )

    return response["message"]["content"]

In [25]:
def evaluator_agent(user_input, response):
    short_response = response[:300]

    messages = [
        {"role": "system", "content": EVALUATOR_PROMPT},
        {"role": "user", "content": f"""
Question: {user_input}

Response: {short_response}
"""}
    ]

    eval_response = ollama.chat(
        model=EVALUATOR_MODEL,
        messages=messages
    )

    content = eval_response["message"]["content"]

    try:
        json_str = re.search(r"\{.*\}", content, re.DOTALL).group()
        return json_str
    except:
        return '{"score": 5, "verdict": "approve", "feedback": "fallback", "improved_response": ""}'

In [26]:
def multi_agent_chat(user_input, chat_history=[]):
    # Step 1: Generate
    raw_response = generator_agent(user_input, chat_history)

    # Step 2: Evaluate
    evaluation = evaluator_agent(user_input, raw_response)

    try:
        eval_json = json.loads(evaluation)
    except:
        eval_json = {
            "score": "N/A",
            "verdict": "approve",
            "feedback": "JSON parsing failed",
            "improved_response": raw_response
        }

    # Step 3: Decision Logic (SMART)
    improved = eval_json.get("improved_response", "").strip()

    if len(raw_response.split()) > 80:
        final_response = improved if improved and improved != raw_response else raw_response

    elif improved:
        final_response = improved

    else:
        final_response = raw_response

    # Step 4: Clean duplicates
    final_response = clean_response(final_response)

    # Step 5: Update history
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": final_response})

    # Step 6: Debug output
    print("\n🧠 Generator Response:\n", raw_response)
    print("\n🔍 Evaluator Output:\n", eval_json)
    print("\n✅ Final Response:\n", final_response)

    return final_response, chat_history

In [ ]:
#chat_history = []

#reply, chat_history = multi_agent_chat(
    #"can you discuss your role as a president of tht club that you been part of in college? ",
    #chat_history
#)

#print("\n🎯 OUTPUT:\n", reply)


🧠 Generator Response:
 I don't recall being a President of any club in college. I was a Graduate Engineer at Microland Ltd after completing my studies, and I didn't have a role as a president of any student organization. My focus shifted to building and deploying scalable AI solutions in cloud environments from that point on.

🔍 Evaluator Output:
 {'score': 8, 'verdict': 'improve', 'feedback': 'The response is clear and factual, but could be more engaging. Consider adding a brief anecdote or storytelling element to make the answer more relatable and human.', 'improved_response': "As I reflect on my college experience, I remember being an active member of a few student organizations. Although I didn't hold the presidency of any club, I did take on leadership roles in various capacities. After completing my studies, my focus shifted towards building scalable AI solutions in cloud environments. I'm proud to say that my passion for innovation continued to drive me forward."}

✅ Final Resp

In [27]:
import gradio as gr

# Wrapper for Gradio
def gradio_chat(user_message, history):
    if history is None:
        history = []

    # Convert Gradio history → your format
    chat_history = []
    for human, bot in history:
        chat_history.append({"role": "user", "content": human})
        chat_history.append({"role": "assistant", "content": bot})

    # Call your multi-agent system
    reply, updated_history = multi_agent_chat(user_message, chat_history)

    return reply


# Launch UI
demo = gr.ChatInterface(
    fn=gradio_chat,
    title="🧠 PersonaForge - AI Persona Assistant",
    description="Talk to an AI that represents my professional experience and background.",
)

demo.launch()

e:\PersonaForge-A-Multi-Agent-LLM-System-for-Intelligent-Interaction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.



🧠 Generator Response:
 I'm Nikith, a Cloud AI Engineer with AWS Certified Machine Learning Engineer – Associate (MLA-C01) certification. I help organizations build and deploy scalable, production-grade AI solutions on AWS. Let's connect if you're exploring AI/Cloud/LLM space or need expertise in these areas.

🔍 Evaluator Output:
 {'score': 8, 'verdict': 'approve', 'feedback': 'Clear and concise introduction, but consider making it more conversational to establish a connection with the user.', 'improved_response': "Hi, I'm Nikith - a Cloud AI Engineer with AWS Certified Machine Learning Engineer – Associate (MLA-C01) certification. If you're exploring AI/Cloud or need help building scalable AI solutions on AWS, let's connect!"}

✅ Final Response:
 Hi, I'm Nikith - a Cloud AI Engineer with AWS Certified Machine Learning Engineer – Associate (MLA-C01) certification. If you're exploring AI/Cloud or need help building scalable AI solutions on AWS, let's connect!
